In [140]:
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.engine import URL
import os

### 📚 Data Consolidation
> It is the process of combining multiple datasets into single, unified dataframe for streamlined analysis.<br>
> It can be done through pd.concat() or pd.merge()

In [141]:
path=r"C:\Users\Prateek\Desktop\Retail Intelligence and Customer Analytics\Dataset\Online_Retail_II_by_Daqing_Chen\csv"
file_list=os.listdir(r"C:\Users\Prateek\Desktop\Retail Intelligence and Customer Analytics\Dataset\Online_Retail_II_by_Daqing_Chen\csv")
df=pd.DataFrame()
for i in file_list:
    if i.endswith('.csv'):
        final_path=os.path.join(path,i)
        df1=pd.read_csv(final_path)
        i=i.lstrip('online_retail_')
        df1['TimePeriod']=i.strip('.csv')
        df=pd.concat([df,df1], ignore_index=True)

In [142]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TimePeriod
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,12/1/2009 7:45,6.95,13085.0,United Kingdom,2009-2010
1,489434,79323P,PINK CHERRY LIGHTS,12,12/1/2009 7:45,6.75,13085.0,United Kingdom,2009-2010
2,489434,79323W,WHITE CHERRY LIGHTS,12,12/1/2009 7:45,6.75,13085.0,United Kingdom,2009-2010
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,12/1/2009 7:45,2.10,13085.0,United Kingdom,2009-2010
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,12/1/2009 7:45,1.25,13085.0,United Kingdom,2009-2010


In [143]:
rows, cols=df.shape
print(f"Total Rows: {rows:,}")
print("Total Columns: ", cols)

Total Rows: 1,067,371
Total Columns:  9


#### ➡️ Exporting DataFrame to SQL for reconciliation analysis
💡 Note: before exporting to SQL create database in server.
##### ▶️ MySQL query for creating a database in mysql:
            create database if not exists retail;

In [144]:
db_url = URL.create(
    'mysql+mysqlconnector',
    username='root',
    password='prateek@123',
    host='localhost',
    database='retail'
) # for creating database url

eng = create_engine(db_url) # creating sqlalchemy engine object

with eng.connect() as conn:
    print("Connection established successfully!!!")

Connection established successfully!!!


💡 Note:- It will ignore the cell during runtime. <br>
`%%script false --no-raise-error`

In [145]:
%%script false --no-raise-error
with eng.begin() as connection: # this ensures to auto-commit and safely close the connection.
    df.to_sql(
        name='retailcombined', # table_name
        if_exists='replace',
        con=connection, # adbc connector tells where to send data
        index=False,
        chunksize=10000, # Insert in batches (helpful in large DataFrames)
        method='multi' # passes multiple values in query
    )
    print("Data-export successful!!!")

Couldn't find program: 'false'


💡 Note: Use chunksize and method parameters to avoid connection abort error.

In [146]:
df['InvoiceDate']=pd.to_datetime(df['InvoiceDate'], format='%m/%d/%Y %H:%M') # converting 'InvoiceDate' datatype to datetime
df['CustomerID']=df['CustomerID'].astype('Int64') # converting 'CustomerID' datatype to Int64

In [147]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 9 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   InvoiceNo    1067371 non-null  str           
 1   StockCode    1067371 non-null  str           
 2   Description  1062989 non-null  str           
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[us]
 5   UnitPrice    1067371 non-null  float64       
 6   CustomerID   824364 non-null   Int64         
 7   Country      1067371 non-null  str           
 8   TimePeriod   1067371 non-null  str           
dtypes: Int64(1), datetime64[us](1), float64(1), int64(1), str(5)
memory usage: 74.3 MB


In [148]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TimePeriod
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,2009-2010
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,2009-2010
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,2009-2010
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,2009-2010
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,2009-2010


In [149]:
# finding number of duplicate rows
dup=df.duplicated().sum()
print(f"Duplicate rows: {dup:,}")

Duplicate rows: 12,133


In [150]:
# dropping duplicates
df.drop_duplicates(inplace=True)
rows, cols=df.shape
print(f"Total Rows: {rows:,}")
print("Total Columns: ", cols)

Total Rows: 1,055,238
Total Columns:  9


In [151]:
df.isnull().sum()

InvoiceNo           0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     242870
Country             0
TimePeriod          0
dtype: int64

In [152]:
# Null CustomerID's and Negative quantity values
cust_null = df.CustomerID.isnull().sum()
negative_quant = df[df['Quantity']<0].shape[0]

print(f"Null CustomerID records: {cust_null:,}")
print(f"Negative Quantity records: {negative_quant:,}")

Null CustomerID records: 242,870
Negative Quantity records: 22,889


In [153]:
# Invoices that doesn't match the invoice pattern
pattern=r'^[cC]?\d{6}$'
invalid_invoices = df[~df['InvoiceNo'].str.match(pattern)]
invalid_invoices

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TimePeriod
179403,A506401,B,Adjust bad debt,1,2010-04-29 13:36:00,-53594.36,<NA>,United Kingdom,2009-2010
276274,A516228,B,Adjust bad debt,1,2010-07-19 11:24:00,-44031.79,<NA>,United Kingdom,2009-2010
403472,A528059,B,Adjust bad debt,1,2010-10-20 12:04:00,-38925.87,<NA>,United Kingdom,2009-2010
825443,A563185,B,Adjust bad debt,1,2011-08-12 14:50:00,11062.06,<NA>,United Kingdom,2010-2011
825444,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,<NA>,United Kingdom,2010-2011
825445,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,<NA>,United Kingdom,2010-2011


In [154]:
df['Amount'] = df['UnitPrice'] * df['Quantity'] # adding a column total purchase amount as amount
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TimePeriod,Amount
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,2009-2010,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,2009-2010,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,2009-2010,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,2009-2010,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,2009-2010,30.0


In [155]:
# filtering null CustomerID and negative Quantity
rfm_data =  df[(df['CustomerID'].notna()) & (df['Quantity']>=0)]
rfm_data.shape[0]

793680

In [156]:
min_date=df['InvoiceDate'].min()
max_date=df['InvoiceDate'].max()
snapshot_date = max_date + pd.DateOffset(days=1) # used for calculations
print(f"Dataset date range is from {min_date} to {max_date}")

Dataset date range is from 2009-12-01 07:45:00 to 2011-12-09 12:50:00


#### ➡️ Calculating 'Recency', 'Frequency' and 'Monetary' of each customer.

In [157]:
rfm = rfm_data.groupby('CustomerID').agg(
    Recency=('InvoiceDate', lambda x: (snapshot_date - x.max()).days),
    Frequency=('InvoiceNo','nunique'),
    Monetary=('Amount','sum')
).reset_index()
rfm

,CustomerID,Recency,Frequency,Monetary
0,12346,326,12,77556.46
1,12347,2,8,5633.32
2,12348,75,5,2019.40
3,12349,19,4,4428.69
4,12350,310,1,334.40
...,...,...,...,...
5876,18283,4,22,2664.90
5877,18284,432,1,461.68
5878,18285,661,1,427.00
5879,18286,477,2,1296.43


In [158]:
# Assigning RFM score
# 0 --> Lowest score
# 5 --> Highest score

rfm['R_Score'] = pd.qcut(rfm.Recency, q=5, labels=[5,4,3,2,1]).astype(int)
rfm['F_Score'] = pd.qcut(rfm.Frequency.rank(method='first'), q=5, labels=[1,2,3,4,5]).astype(int)
rfm['M_Score'] = pd.qcut(rfm.Monetary, q=5, labels=[1,2,3,4,5]).astype(int)

rfm.head(20)

,CustomerID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score
0,12346,326,12,77556.46,2,5,5
1,12347,2,8,5633.32,5,4,5
2,12348,75,5,2019.40,3,4,4
3,12349,19,4,4428.69,5,3,5
4,12350,310,1,334.40,2,1,2
5,12351,375,1,300.93,2,1,2
6,12352,36,10,2849.84,4,5,4
7,12353,204,2,406.76,2,2,2
8,12354,232,1,1079.40,2,1,3
9,12355,214,2,947.61,2,2,3


In [159]:
def segmentation(df):
    if df['R_Score']>=4 and df['F_Score']>=4 and df['M_Score']>=4:
        return 'Champions'
    elif df['R_Score']>=3 and df['F_Score']>=3:
        return 'Loyal'
    elif df['R_Score']<=3 and df['F_Score']>=3:
        return 'At Risk'
    elif df['R_Score']>=4 and df['F_Score']<=2:
        return 'New'
    elif df['R_Score']==3 and df['F_Score']<=2:
        return 'Needs Attention'
    else:
        return 'Lost'

rfm['Segmentation'] = rfm.apply(segmentation, axis=1)
rfm.head(20)

,CustomerID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,Segmentation
0,12346,326,12,77556.46,2,5,5,At Risk
1,12347,2,8,5633.32,5,4,5,Champions
2,12348,75,5,2019.40,3,4,4,Loyal
3,12349,19,4,4428.69,5,3,5,Loyal
4,12350,310,1,334.40,2,1,2,Lost
5,12351,375,1,300.93,2,1,2,Lost
6,12352,36,10,2849.84,4,5,4,Champions
7,12353,204,2,406.76,2,2,2,Lost
8,12354,232,1,1079.40,2,1,3,Lost
9,12355,214,2,947.61,2,2,3,Lost


In [160]:
# creating new column IsChurned if the recency is more than 90 days

rfm['IsChurned'] = rfm['Recency'] > 90
rfm.head(20)

,CustomerID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,Segmentation,IsChurned
0,12346,326,12,77556.46,2,5,5,At Risk,True
1,12347,2,8,5633.32,5,4,5,Champions,False
2,12348,75,5,2019.40,3,4,4,Loyal,False
3,12349,19,4,4428.69,5,3,5,Loyal,False
4,12350,310,1,334.40,2,1,2,Lost,True
5,12351,375,1,300.93,2,1,2,Lost,True
6,12352,36,10,2849.84,4,5,4,Champions,False
7,12353,204,2,406.76,2,2,2,Lost,True
8,12354,232,1,1079.40,2,1,3,Lost,True
9,12355,214,2,947.61,2,2,3,Lost,True


In [161]:
# Adding column InvoiceMonth to the DataFrame and merging RFM to main DataFrame

df = df[(df['CustomerID'].notna()) & (df['Quantity']>0)]
df['InvoiceMonth'] = df['InvoiceDate'].dt.month_name()
df = pd.merge(df, rfm, how='left', left_on='CustomerID', right_on='CustomerID')
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TimePeriod,Amount,InvoiceMonth,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,Segmentation,IsChurned
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,2009-2010,83.4,December,158,8,2433.28,3,4,4,Loyal,True
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,2009-2010,81.0,December,158,8,2433.28,3,4,4,Loyal,True
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,2009-2010,81.0,December,158,8,2433.28,3,4,4,Loyal,True
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,2009-2010,100.8,December,158,8,2433.28,3,4,4,Loyal,True
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,2009-2010,30.0,December,158,8,2433.28,3,4,4,Loyal,True


In [162]:
df.shape

(793680, 19)

💡 Note:- It will ignore the cell during runtime. <br>
`%%script false --no-raise-error`

In [163]:
%%script false --no-raise-error
df.to_excel(r"Z:\Retail Intelligence and Customer Analytics\Dataset_Cleaned\Retail_Cleaned.xlsx", engine='xlsxwriter', index=False) # export to excel
print("Data Transferred Successfuly!!!")

Couldn't find program: 'false'
